In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark
sales = spark.createDataFrame(
    [
        ("c1", 100),
        ("c2", 200),
        ("c3", 300),
        ("c4", 400),
        ("c5", 500),
    ],
    ["customer_id", "amount"]
)

customers = spark.createDataFrame(
    [
        ("c1", "East"),
        ("c2", "West"),
        ("c3", "East"),
        ("c4", "North"),
        ("c5", "South"),
    ],
    ["customer_id", "region"]
)
sales.show()
customers.show()

+-----------+------+
|customer_id|amount|
+-----------+------+
|         c1|   100|
|         c2|   200|
|         c3|   300|
|         c4|   400|
|         c5|   500|
+-----------+------+

+-----------+------+
|customer_id|region|
+-----------+------+
|         c1|  East|
|         c2|  West|
|         c3|  East|
|         c4| North|
|         c5| South|
+-----------+------+



# A1. Write partitioned Parquet by year/month from a date column.

In [7]:
# Lets add a date column to our sales dataframe
sales_with_date = (
    sales
    .withColumn(
        "sale_date",
        F.to_date(F.lit("2024-01-15"))
    )
)

sales_with_date.show()

+-----------+------+----------+
|customer_id|amount| sale_date|
+-----------+------+----------+
|         c1|   100|2024-01-15|
|         c2|   200|2024-01-15|
|         c3|   300|2024-01-15|
|         c4|   400|2024-01-15|
|         c5|   500|2024-01-15|
+-----------+------+----------+



In [10]:
sales_partitioned = (
    sales_with_date
    .withColumn("year", F.year("sale_date"))
    .withColumn("month", F.month("sale_date"))
)

sales_partitioned.show()

+-----------+------+----------+----+-----+
|customer_id|amount| sale_date|year|month|
+-----------+------+----------+----+-----+
|         c1|   100|2024-01-15|2024|    1|
|         c2|   200|2024-01-15|2024|    1|
|         c3|   300|2024-01-15|2024|    1|
|         c4|   400|2024-01-15|2024|    1|
|         c5|   500|2024-01-15|2024|    1|
+-----------+------+----------+----+-----+



# A2. Read with partition filter; optional explain.


In [13]:
sales_jan_2024 = (
    spark.read
    .parquet("/tmp/sales_parquet")
    .filter(
        (F.col("year") == 2024) &
        (F.col("month") == 1)
    )
)

sales_jan_2024.show()

+-----------+------+----------+----+-----+
|customer_id|amount| sale_date|year|month|
+-----------+------+----------+----+-----+
|         c3|   300|2024-01-15|2024|    1|
|         c4|   400|2024-01-15|2024|    1|
|         c5|   500|2024-01-15|2024|    1|
|         c1|   100|2024-01-15|2024|    1|
|         c2|   200|2024-01-15|2024|    1|
+-----------+------+----------+----+-----+



# A3. Write MERGE pseudocode for SCD1 customer upsert.
             Source
                |
                v
        Match customer_id
                |
        +-------+-------+
        |               |
      MATCH          NO MATCH
        |               |
        v               v
     UPDATE           INSERT
        |               |
        +-------+-------+
                |
                v
              Target

In [16]:
# i want to create a basic dataframe
source_customers = spark.createDataFrame(
    [
        ("c1", "Rahul", "West"),
        ("c6", "Amit", "North"),
    ],
    ["customer_id", "name", "region"]
)
source_customers.show()

+-----------+-----+------+
|customer_id| name|region|
+-----------+-----+------+
|         c1|Rahul|  West|
|         c6| Amit| North|
+-----------+-----+------+



| Topic                   | One-line interview answer                                                                                     |
| ----------------------- | ------------------------------------------------------------------------------------------------------------- |
| **Partitioned Parquet** | Write using `partitionBy("year", "month")` to organize files by common filter columns.                        |
| **Partition pruning**   | Spark skips irrelevant partition directories when filtering on partition columns.                             |
| **`repartition()`**     | Redistributes data and normally causes a shuffle.                                                             |
| **`partitionBy()`**     | Controls physical directory/file organization when writing.                                                   |
| **SCD1**                | Overwrite the existing dimension record with the latest value; no history.                                    |
| **MERGE**               | Match → update; no match → insert.                                                                            |
| **CSV**                 | Text-based, simple and portable, but less efficient for analytics.                                            |
| **Parquet**             | Columnar, compressed, schema-aware, and efficient for analytical workloads.                                   |
| **Delta Lake**          | Adds transactional/table-management capabilities such as ACID, MERGE, schema enforcement, and time travel.    |
| **OPTIMIZE**            | Improves Delta file layout, commonly through file compaction.                                                 |
| **Z-ORDER**             | Organizes data to improve data skipping for selected query/filter columns.                                    |
| **Liquid clustering**   | A newer Databricks approach worth evaluating for new table designs instead of automatically choosing Z-ORDER. |


In [18]:
s